# Xenia — ResNet fine-tuning experiment

**Owner:** Xenia only.

Same fixed splits and `MultiLabelBinarizer` as `main.py`. Start from the existing **ResNet-50** baseline in `main.py` / `main.ipynb` and iterate here.

In [3]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import shutil
import time

def find_project_root():
    search_roots = [
        Path("/content/drive/MyDrive/ieor142b"),
        Path("/content/drive/MyDrive"),
    ]

    for root in search_roots:
        if (
            (root / "experimentation" / "shared.py").is_file()
            and (root / "cleaned" / "downloaded_posters").is_dir()
            and (root / "cleaned" / "MovieGenre_clean_with_images_full.csv").is_file()
        ):
            return root

    print("Searching MyDrive for ieor142b project folder...")

    for shared_py in Path("/content/drive/MyDrive").rglob("shared.py"):
        possible_root = shared_py.parent.parent
        if (
            (possible_root / "cleaned" / "downloaded_posters").is_dir()
            and (possible_root / "cleaned" / "MovieGenre_clean_with_images_full.csv").is_file()
        ):
            return possible_root

    raise RuntimeError(
        "Could not find project folder. Make sure the shared ieor142b folder "
        "has been added as a shortcut to My Drive."
    )

PROJECT_ROOT = find_project_root()

DRIVE_POSTER_DIR = PROJECT_ROOT / "cleaned" / "downloaded_posters"
LOCAL_POSTER_DIR = Path("/content/downloaded_posters")

print("Project root:", PROJECT_ROOT)
print("Drive poster dir:", DRIVE_POSTER_DIR)
print("Drive poster dir exists:", DRIVE_POSTER_DIR.is_dir())
print("Num posters in Drive:", len(list(DRIVE_POSTER_DIR.glob("*.jpg"))))

start = time.perf_counter()

if LOCAL_POSTER_DIR.exists():
    print("Local poster folder already exists. Skipping copy.")
else:
    print("Copying posters from Drive to /content...")
    shutil.copytree(DRIVE_POSTER_DIR, LOCAL_POSTER_DIR)

elapsed = time.perf_counter() - start

POSTER_DIR = LOCAL_POSTER_DIR

print("Done.")
print("Using local POSTER_DIR:", POSTER_DIR)
print("Local poster dir exists:", POSTER_DIR.is_dir())
print("Num posters local:", len(list(POSTER_DIR.glob("*.jpg"))))
print(f"Copy time: {elapsed / 60:.2f} minutes")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root: /content/drive/MyDrive/ieor142b
Drive poster dir: /content/drive/MyDrive/ieor142b/cleaned/downloaded_posters
Drive poster dir exists: True
Num posters in Drive: 9547
Copying posters from Drive to /content...
Done.
Using local POSTER_DIR: /content/downloaded_posters
Local poster dir exists: True
Num posters local: 9547
Copy time: 6.25 minutes


In [4]:
from pathlib import Path

POSTER_DIR = Path("/content/downloaded_posters")

print("Using POSTER_DIR:", POSTER_DIR)
print("Exists:", POSTER_DIR.is_dir())
print("Num posters:", len(list(POSTER_DIR.glob("*.jpg"))))

Using POSTER_DIR: /content/downloaded_posters
Exists: True
Num posters: 9547


In [6]:
# ============================================================
# FAST CLEANED POSTER TRAINING: RESNET-50 + FOCAL LOSS
# Optimized for Colab runtime:
# - Uses cleaned strict splits from shared.py
# - Uses local /content/downloaded_posters if available
# - Copies from Drive only if local posters are missing
# - Uses mixed precision on GPU
# - Keeps only ResNet layer4 + classifier head trainable
# - Prints batch progress + epoch timing + ETA
# ============================================================

import os
import sys
import time
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from google.colab import drive
drive.mount("/content/drive")


# ============================================================
# 1. FIND PROJECT ROOT IN GOOGLE DRIVE
# ============================================================

def find_project_root() -> Path:
    likely_roots = [
        Path("/content/drive/MyDrive/ieor142b"),
        Path("/content/drive/MyDrive/142b/ieor142b"),
        Path("/content/drive/MyDrive/project"),
    ]

    for root in likely_roots:
        if (
            (root / "experimentation" / "shared.py").is_file()
            and (root / "cleaned" / "MovieGenre_clean_with_images_full.csv").is_file()
            and (root / "cleaned" / "downloaded_posters").is_dir()
            and (root / "splits_cleaned" / "train_rows.csv").is_file()
        ):
            return root

    print("Could not find project in likely locations. Searching MyDrive...")

    for shared_py in Path("/content/drive/MyDrive").rglob("shared.py"):
        root = shared_py.parent.parent
        if (
            (root / "cleaned" / "MovieGenre_clean_with_images_full.csv").is_file()
            and (root / "cleaned" / "downloaded_posters").is_dir()
            and (root / "splits_cleaned" / "train_rows.csv").is_file()
        ):
            return root

    raise RuntimeError(
        "Could not find ieor142b project folder. "
        "Make sure the shared folder is added as a shortcut to My Drive."
    )


PROJECT_ROOT = find_project_root()
_ROOT = PROJECT_ROOT

print("PROJECT_ROOT:", PROJECT_ROOT)
print("shared.py exists:", (PROJECT_ROOT / "experimentation" / "shared.py").is_file())
print("cleaned CSV exists:", (PROJECT_ROOT / "cleaned" / "MovieGenre_clean_with_images_full.csv").is_file())
print("Drive poster folder exists:", (PROJECT_ROOT / "cleaned" / "downloaded_posters").is_dir())
print("splits_cleaned exists:", (PROJECT_ROOT / "splits_cleaned").is_dir())


# ============================================================
# 2. IMPORT SHARED.PY + LOAD STRICT CLEANED SPLITS
# ============================================================

sys.path.insert(0, str(PROJECT_ROOT / "experimentation"))

# Avoid stale imports if rerunning
if "shared" in sys.modules:
    del sys.modules["shared"]

from shared import (
    STRICT_BASELINE as BASELINE,
    STRICT_POSTER_DIR as DRIVE_POSTER_DIR,
    load_strict_split_dataframes,
    set_seed,
    save_metrics_json,
)

set_seed(BASELINE["seed"])

train_df, val_df, test_df, mlb = load_strict_split_dataframes()
num_genres = len(mlb.classes_)

print("\nLoaded strict cleaned splits.")
print("Train / Val / Test:", len(train_df), len(val_df), len(test_df))
print("Num genres:", num_genres)


# ============================================================
# 3. USE LOCAL /content POSTERS
# ============================================================

DRIVE_POSTER_DIR = PROJECT_ROOT / "cleaned" / "downloaded_posters"
LOCAL_POSTER_DIR = Path("/content/downloaded_posters")

print("\nDrive poster dir:", DRIVE_POSTER_DIR)
print("Drive posters exist:", DRIVE_POSTER_DIR.is_dir())

if not LOCAL_POSTER_DIR.exists():
    print("Local posters not found. Copying Drive posters to /content...")
    files = list(DRIVE_POSTER_DIR.glob("*.jpg"))
    LOCAL_POSTER_DIR.mkdir(parents=True, exist_ok=True)

    start_copy = time.perf_counter()

    for src in tqdm(files, desc="Copying posters to /content"):
        dst = LOCAL_POSTER_DIR / src.name
        if not dst.exists():
            shutil.copy2(src, dst)

    copy_elapsed = time.perf_counter() - start_copy
    print(f"Copy done in {copy_elapsed / 60:.2f} minutes.")
else:
    print("Local poster folder already exists. Skipping copy.")

POSTER_DIR = LOCAL_POSTER_DIR

print("Using local POSTER_DIR:", POSTER_DIR)
print("Local poster dir exists:", POSTER_DIR.is_dir())
print("Num local posters:", len(list(POSTER_DIR.glob('*.jpg'))))


# ============================================================
# 4. CONFIG: FAST BUT STILL ACCURATE
# ============================================================

class XeniaConfig:
    IMG_SIZE = BASELINE["img_size"]        # 224
    BATCH_SIZE = 64                        # use 32 if GPU runs out of memory
    NUM_EPOCHS = 16                        # lower than 30 for speed
    LR = 1e-4
    WEIGHT_DECAY = 1e-4
    DROPOUT = 0.5
    PATIENCE = 3                           # early stop faster
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    SEED = BASELINE["seed"]
    FOCAL_ALPHA = BASELINE["focal_alpha"]
    FOCAL_GAMMA = BASELINE["focal_gamma"]
    THRESHOLD = BASELINE["metric_threshold"]

    BEST_MODEL_PATH = PROJECT_ROOT / "results" / "xenia_best_resnet50_fast.pt"
    METRICS_PATH = PROJECT_ROOT / "results" / "xenia_resnet50_fast_metrics.json"


torch.manual_seed(XeniaConfig.SEED)
np.random.seed(XeniaConfig.SEED)

# Speeds up fixed-size image convolutions on GPU
torch.backends.cudnn.benchmark = True

print("\nUsing device:", XeniaConfig.DEVICE)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
print("Batch size:", XeniaConfig.BATCH_SIZE)
print("Epochs:", XeniaConfig.NUM_EPOCHS)


# ============================================================
# 5. DATASET
# ============================================================

class MoviePosterDataset(Dataset):
    def __init__(self, df, mlb, poster_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.mlb = mlb
        self.poster_dir = Path(poster_dir)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def _poster_path(self, row):
        imdb_id = str(int(float(row["imdbId"])))
        return self.poster_dir / f"{imdb_id}.jpg"

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self._poster_path(row)

        try:
            img = Image.open(img_path).convert("RGB")
        except Exception:
            # Fallback should be rare if strict cleaned data is correct
            img = Image.new(
                "RGB",
                (XeniaConfig.IMG_SIZE, XeniaConfig.IMG_SIZE),
                (128, 128, 128),
            )

        if self.transform:
            img = self.transform(img)

        genres = row["Genre"].split("|")
        label = torch.tensor(
            self.mlb.transform([genres])[0],
            dtype=torch.float32,
        )

        return img, label


def count_missing_posters(df, poster_dir):
    poster_dir = Path(poster_dir)
    missing = 0

    for _, row in df.iterrows():
        imdb_id = str(int(float(row["imdbId"])))
        if not (poster_dir / f"{imdb_id}.jpg").is_file():
            missing += 1

    return missing


print("\nChecking missing posters...")
print("Missing train posters:", count_missing_posters(train_df, POSTER_DIR))
print("Missing val posters:", count_missing_posters(val_df, POSTER_DIR))
print("Missing test posters:", count_missing_posters(test_df, POSTER_DIR))


# ============================================================
# 6. TRANSFORMS
# ============================================================

def get_transforms():
    mean = [0.485, 0.456, 0.406]
    std = [0.229, 0.224, 0.225]

    train_tf = transforms.Compose([
        transforms.Resize((XeniaConfig.IMG_SIZE + 32, XeniaConfig.IMG_SIZE + 32)),
        transforms.RandomCrop(XeniaConfig.IMG_SIZE),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    val_tf = transforms.Compose([
        transforms.Resize((XeniaConfig.IMG_SIZE, XeniaConfig.IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    return train_tf, val_tf


# ============================================================
# 7. FOCAL LOSS
# ============================================================

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        bce_loss = nn.functional.binary_cross_entropy_with_logits(
            logits,
            targets,
            reduction="none",
        )

        probs = torch.sigmoid(logits)
        p_t = probs * targets + (1 - probs) * (1 - targets)

        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        focal_weight = alpha_t * (1 - p_t) ** self.gamma

        loss = focal_weight * bce_loss

        if self.reduction == "mean":
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        return loss


# ============================================================
# 8. MODEL: RESNET-50, ONLY LAYER4 + HEAD TRAINABLE
# ============================================================

class PosterGenreClassifier(nn.Module):
    def __init__(self, num_genres, dropout=XeniaConfig.DROPOUT):
        super().__init__()

        backbone = models.resnet50(
            weights=models.ResNet50_Weights.IMAGENET1K_V2
        )

        # Freeze full backbone
        for param in backbone.parameters():
            param.requires_grad = False

        # Train only final ResNet block for accuracy/speed balance
        for param in backbone.layer4.parameters():
            param.requires_grad = True

        in_features = backbone.fc.in_features
        backbone.fc = nn.Identity()

        self.backbone = backbone
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_features, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, num_genres),
        )

    def forward(self, x):
        features = self.backbone(x)
        logits = self.head(features)
        return logits


# ============================================================
# 9. METRICS
# ============================================================

def compute_metrics(logits, targets, threshold=XeniaConfig.THRESHOLD):
    preds = (torch.sigmoid(logits) >= threshold).float()

    tp = (preds * targets).sum(dim=1)
    fp = (preds * (1 - targets)).sum(dim=1)
    fn = ((1 - preds) * targets).sum(dim=1)

    precision = (tp / (tp + fp + 1e-8)).mean().item()
    recall = (tp / (tp + fn + 1e-8)).mean().item()
    f1 = (2 * tp / (2 * tp + fp + fn + 1e-8)).mean().item()
    exact = (preds == targets).all(dim=1).float().mean().item()

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "exact_match": exact,
    }


# ============================================================
# 10. DATASETS + LOADERS
# ============================================================

train_tf, val_tf = get_transforms()

train_ds = MoviePosterDataset(train_df, mlb, POSTER_DIR, train_tf)
val_ds = MoviePosterDataset(val_df, mlb, POSTER_DIR, val_tf)
test_ds = MoviePosterDataset(test_df, mlb, POSTER_DIR, val_tf)

pin_memory = XeniaConfig.DEVICE == "cuda"

# With posters in /content, 2 workers should be safe and faster.
# If this hangs, change NUM_WORKERS to 0 and rerun from this cell.
NUM_WORKERS = 2

loader_kwargs = {
    "num_workers": NUM_WORKERS,
    "pin_memory": pin_memory,
}

if NUM_WORKERS > 0:
    loader_kwargs["persistent_workers"] = True
    loader_kwargs["prefetch_factor"] = 2

train_loader = DataLoader(
    train_ds,
    batch_size=XeniaConfig.BATCH_SIZE,
    shuffle=True,
    **loader_kwargs,
)

val_loader = DataLoader(
    val_ds,
    batch_size=XeniaConfig.BATCH_SIZE,
    shuffle=False,
    **loader_kwargs,
)

test_loader = DataLoader(
    test_ds,
    batch_size=XeniaConfig.BATCH_SIZE,
    shuffle=False,
    **loader_kwargs,
)

print("\nTrain batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))


# Quick batch sanity check
print("\nTesting one batch load...")
start = time.perf_counter()
imgs, labels = next(iter(train_loader))
elapsed = time.perf_counter() - start

print("One batch load time:", round(elapsed, 3), "seconds")
print("Batch image shape:", imgs.shape)
print("Batch label shape:", labels.shape)


# ============================================================
# 11. TRAIN/EVAL FUNCTIONS WITH MIXED PRECISION
# ============================================================

# New PyTorch AMP API with fallback to old behavior
use_amp = XeniaConfig.DEVICE == "cuda"
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    all_logits = []
    all_targets = []

    for imgs, labels in tqdm(loader, desc="Training batches", leave=True):
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = model(imgs)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * imgs.size(0)
        all_logits.append(logits.detach().cpu())
        all_targets.append(labels.detach().cpu())

    avg_loss = total_loss / len(loader.dataset)
    metrics = compute_metrics(torch.cat(all_logits), torch.cat(all_targets))

    return avg_loss, metrics


@torch.no_grad()
def evaluate(model, loader, criterion, device, desc="Validation batches"):
    model.eval()

    total_loss = 0.0
    all_logits = []
    all_targets = []

    for imgs, labels in tqdm(loader, desc=desc, leave=True):
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = model(imgs)
            loss = criterion(logits, labels)

        total_loss += loss.item() * imgs.size(0)
        all_logits.append(logits.cpu())
        all_targets.append(labels.cpu())

    avg_loss = total_loss / len(loader.dataset)
    metrics = compute_metrics(torch.cat(all_logits), torch.cat(all_targets))

    return avg_loss, metrics


# ============================================================
# 12. MODEL, LOSS, OPTIMIZER, SCHEDULER
# ============================================================

model = PosterGenreClassifier(num_genres).to(XeniaConfig.DEVICE)

criterion = FocalLoss(
    alpha=XeniaConfig.FOCAL_ALPHA,
    gamma=XeniaConfig.FOCAL_GAMMA,
)

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=XeniaConfig.LR,
    weight_decay=XeniaConfig.WEIGHT_DECAY,
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=XeniaConfig.NUM_EPOCHS,
)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print("\nModel ready.")
print(f"Trainable params: {trainable_params:,}")
print(f"Total params: {total_params:,}")


# ============================================================
# 13. TRAINING LOOP
# ============================================================

history = {
    "train_loss": [],
    "val_loss": [],
    "train_f1": [],
    "val_f1": [],
    "val_precision": [],
    "val_recall": [],
}

best_val_f1 = 0.0
patience_ctr = 0
epochs_run = 0
epoch_times = []

XeniaConfig.BEST_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

print("\n── Fast Training Xenia ResNet-50 ──────────────────────────────")

total_start = time.perf_counter()

for epoch in range(1, XeniaConfig.NUM_EPOCHS + 1):
    epoch_start = time.perf_counter()
    epochs_run = epoch

    train_loss, train_m = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        XeniaConfig.DEVICE,
    )

    val_loss, val_m = evaluate(
        model,
        val_loader,
        criterion,
        XeniaConfig.DEVICE,
        desc="Validation batches",
    )

    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_f1"].append(train_m["f1"])
    history["val_f1"].append(val_m["f1"])
    history["val_precision"].append(val_m["precision"])
    history["val_recall"].append(val_m["recall"])

    epoch_time = time.perf_counter() - epoch_start
    epoch_times.append(epoch_time)

    avg_epoch_time = sum(epoch_times) / len(epoch_times)
    eta_seconds = avg_epoch_time * (XeniaConfig.NUM_EPOCHS - epoch)
    elapsed_seconds = time.perf_counter() - total_start

    print(
        f"Epoch {epoch:02d}/{XeniaConfig.NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val F1: {val_m['f1']:.4f} | "
        f"Val Exact: {val_m['exact_match']:.4f} | "
        f"Epoch Time: {epoch_time / 60:.1f} min | "
        f"Elapsed: {elapsed_seconds / 60:.1f} min | "
        f"ETA: {eta_seconds / 60:.1f} min"
    )

    if val_m["f1"] > best_val_f1:
        best_val_f1 = val_m["f1"]
        torch.save(model.state_dict(), XeniaConfig.BEST_MODEL_PATH)
        patience_ctr = 0
        print("  Saved new best model.")
    else:
        patience_ctr += 1
        print(f"  No improvement. Patience: {patience_ctr}/{XeniaConfig.PATIENCE}")

        if patience_ctr >= XeniaConfig.PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break


# ============================================================
# 14. TEST EVALUATION + SAVE METRICS
# ============================================================

print("\n── Test Set Evaluation ──────────────────────────────")

model.load_state_dict(
    torch.load(XeniaConfig.BEST_MODEL_PATH, map_location=XeniaConfig.DEVICE)
)

test_loss, test_m = evaluate(
    model,
    test_loader,
    criterion,
    XeniaConfig.DEVICE,
    desc="Test batches",
)

print(f"Test Loss:        {test_loss:.4f}")
print(f"Test F1:          {test_m['f1']:.4f}")
print(f"Test Precision:   {test_m['precision']:.4f}")
print(f"Test Recall:      {test_m['recall']:.4f}")
print(f"Test Exact Match: {test_m['exact_match']:.4f}")

metrics_payload = {
    "model_name": "xenia_resnet50_focal_loss_fast_cleaned_local_posters",
    "seed": XeniaConfig.SEED,
    "img_size": XeniaConfig.IMG_SIZE,
    "train_size": len(train_df),
    "val_size": len(val_df),
    "test_size": len(test_df),
    "best_val_f1": float(best_val_f1),
    "test_f1": float(test_m["f1"]),
    "test_precision": float(test_m["precision"]),
    "test_recall": float(test_m["recall"]),
    "test_exact_match": float(test_m["exact_match"]),
    "num_epochs_run": int(epochs_run),
    "notes": (
        "Fast ResNet-50 run on strict cleaned dataset. "
        "Uses local /content/downloaded_posters, mixed precision, batch size 64, "
        "and trains only ResNet layer4 plus classification head."
    ),
}

save_metrics_json(XeniaConfig.METRICS_PATH, metrics_payload)

print("\nSaved best model to:", XeniaConfig.BEST_MODEL_PATH)
print("Saved metrics to:", XeniaConfig.METRICS_PATH)
print("Done.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_ROOT: /content/drive/MyDrive/ieor142b
shared.py exists: True
cleaned CSV exists: True
Drive poster folder exists: True
splits_cleaned exists: True

Loaded strict cleaned splits.
Train / Val / Test: 8700 1087 1088
Num genres: 24

Drive poster dir: /content/drive/MyDrive/ieor142b/cleaned/downloaded_posters
Drive posters exist: True
Local poster folder already exists. Skipping copy.
Using local POSTER_DIR: /content/downloaded_posters
Local poster dir exists: True
Num local posters: 9547

Using device: cuda
Tesla T4
Batch size: 64
Epochs: 16

Checking missing posters...
Missing train posters: 659
Missing val posters: 78
Missing test posters: 89

Train batches: 136
Val batches: 17
Test batches: 17

Testing one batch load...
One batch load time: 1.597 seconds
Batch image shape: torch.Size([64, 3, 224, 224])
Batch label shape: torch.Size([64, 24])


/tmp/ipykernel_21941/3768282456.py:427: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)



Model ready.
Trainable params: 16,026,136
Total params: 24,569,432

── Fast Training Xenia ResNet-50 ──────────────────────────────


Training batches:   0%|          | 0/136 [00:00<?, ?it/s]

/tmp/ipykernel_21941/3768282456.py:442: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


...

Training batches:   0%|          | 0/136 [00:00<?, ?it/s]

Validation batches:   0%|          | 0/17 [00:00<?, ?it/s]

Epoch 16/16 | Train Loss: 0.0099 | Val Loss: 0.0267 | Val F1: 0.4163 | Val Exact: 0.1141 | Epoch Time: 0.9 min | Elapsed: 17.5 min | ETA: 0.0 min
  Saved new best model.

── Test Set Evaluation ──────────────────────────────


Test batches:   0%|          | 0/17 [00:00<?, ?it/s]

Test Loss:        0.0293
Test F1:          0.3885
Test Precision:   0.5306
Test Recall:      0.3388
Test Exact Match: 0.1039

Saved best model to: /content/drive/MyDrive/ieor142b/results/xenia_best_resnet50_fast.pt
Saved metrics to: /content/drive/MyDrive/ieor142b/results/xenia_resnet50_fast_metrics.json
Done.


In [8]:
# ============================================================
# SECOND RUN: FINE-TUNE FROM BEST FAST RESNET-50 CHECKPOINT
# Goal: improve validation/test performance without huge runtime
# ============================================================

import time
from pathlib import Path

import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm.auto import tqdm


# ─────────────────────────────────────────────
# 1. FINE-TUNE CONFIG
# ─────────────────────────────────────────────
class FineTuneConfig:
    IMG_SIZE = BASELINE["img_size"]
    BATCH_SIZE = 32          # safer because layer3 is now trainable
    NUM_EPOCHS = 6           # short fine-tuning run
    PATIENCE = 2

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    SEED = BASELINE["seed"]
    THRESHOLD = BASELINE["metric_threshold"]

    FOCAL_ALPHA = BASELINE["focal_alpha"]
    FOCAL_GAMMA = BASELINE["focal_gamma"]

    # Load previous best fast model
    PREV_BEST_MODEL_PATH = PROJECT_ROOT / "results" / "xenia_best_resnet50_fast.pt"

    # Save new fine-tuned model separately
    BEST_MODEL_PATH = PROJECT_ROOT / "results" / "xenia_best_resnet50_finetuned.pt"
    METRICS_PATH = PROJECT_ROOT / "results" / "xenia_resnet50_finetuned_metrics.json"


print("Using device:", FineTuneConfig.DEVICE)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
print("Previous checkpoint exists:", FineTuneConfig.PREV_BEST_MODEL_PATH.exists())
print("Previous checkpoint:", FineTuneConfig.PREV_BEST_MODEL_PATH)


# ─────────────────────────────────────────────
# 2. RECREATE DATALOADERS WITH SAFER BATCH SIZE
# ─────────────────────────────────────────────
pin_memory = FineTuneConfig.DEVICE == "cuda"
NUM_WORKERS = 2

loader_kwargs = {
    "num_workers": NUM_WORKERS,
    "pin_memory": pin_memory,
}

if NUM_WORKERS > 0:
    loader_kwargs["persistent_workers"] = True
    loader_kwargs["prefetch_factor"] = 2

train_loader = DataLoader(
    train_ds,
    batch_size=FineTuneConfig.BATCH_SIZE,
    shuffle=True,
    **loader_kwargs,
)

val_loader = DataLoader(
    val_ds,
    batch_size=FineTuneConfig.BATCH_SIZE,
    shuffle=False,
    **loader_kwargs,
)

test_loader = DataLoader(
    test_ds,
    batch_size=FineTuneConfig.BATCH_SIZE,
    shuffle=False,
    **loader_kwargs,
)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))


# ─────────────────────────────────────────────
# 3. LOAD BEST FAST MODEL
# ─────────────────────────────────────────────
model = PosterGenreClassifier(num_genres).to(FineTuneConfig.DEVICE)

model.load_state_dict(
    torch.load(
        FineTuneConfig.PREV_BEST_MODEL_PATH,
        map_location=FineTuneConfig.DEVICE,
    )
)

print("Loaded previous best fast model.")


# ─────────────────────────────────────────────
# 4. UNFREEZE LAYER3 + LAYER4 + HEAD ONLY
# ─────────────────────────────────────────────
# Freeze everything first
for param in model.parameters():
    param.requires_grad = False

# Unfreeze layer3 and layer4 of ResNet backbone
for param in model.backbone.layer3.parameters():
    param.requires_grad = True

for param in model.backbone.layer4.parameters():
    param.requires_grad = True

# Unfreeze classifier head
for param in model.head.parameters():
    param.requires_grad = True

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"Trainable params: {trainable_params:,}")
print(f"Total params: {total_params:,}")


# ─────────────────────────────────────────────
# 5. LOSS + OPTIMIZER WITH DIFFERENT LRS
# ─────────────────────────────────────────────
criterion = FocalLoss(
    alpha=FineTuneConfig.FOCAL_ALPHA,
    gamma=FineTuneConfig.FOCAL_GAMMA,
)

optimizer = optim.AdamW(
    [
        {
            "params": model.backbone.layer3.parameters(),
            "lr": 1e-5,
        },
        {
            "params": model.backbone.layer4.parameters(),
            "lr": 2e-5,
        },
        {
            "params": model.head.parameters(),
            "lr": 5e-5,
        },
    ],
    weight_decay=1e-4,
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=FineTuneConfig.NUM_EPOCHS,
)

use_amp = FineTuneConfig.DEVICE == "cuda"
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


# ─────────────────────────────────────────────
# 6. TRAIN/EVAL FUNCTIONS FOR FINE-TUNE
# ─────────────────────────────────────────────
def finetune_train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    all_logits = []
    all_targets = []

    for imgs, labels in tqdm(loader, desc="Fine-tune training", leave=True):
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = model(imgs)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * imgs.size(0)
        all_logits.append(logits.detach().cpu())
        all_targets.append(labels.detach().cpu())

    avg_loss = total_loss / len(loader.dataset)
    metrics = compute_metrics(torch.cat(all_logits), torch.cat(all_targets))

    return avg_loss, metrics


@torch.no_grad()
def finetune_evaluate(model, loader, criterion, device, desc="Fine-tune validation"):
    model.eval()

    total_loss = 0.0
    all_logits = []
    all_targets = []

    for imgs, labels in tqdm(loader, desc=desc, leave=True):
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = model(imgs)
            loss = criterion(logits, labels)

        total_loss += loss.item() * imgs.size(0)
        all_logits.append(logits.cpu())
        all_targets.append(labels.cpu())

    avg_loss = total_loss / len(loader.dataset)
    metrics = compute_metrics(torch.cat(all_logits), torch.cat(all_targets))

    return avg_loss, metrics


# ─────────────────────────────────────────────
# 7. FINE-TUNING LOOP
# ─────────────────────────────────────────────
history_ft = {
    "train_loss": [],
    "val_loss": [],
    "train_f1": [],
    "val_f1": [],
    "val_precision": [],
    "val_recall": [],
}

best_val_f1 = -1.0
best_val_loss = float("inf")
patience_ctr = 0
epochs_run = 0
epoch_times = []

FineTuneConfig.BEST_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

print("\n── Fine-tuning ResNet-50: layer3 + layer4 + head ─────────────────")

total_start = time.perf_counter()

for epoch in range(1, FineTuneConfig.NUM_EPOCHS + 1):
    epoch_start = time.perf_counter()
    epochs_run = epoch

    train_loss, train_m = finetune_train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        FineTuneConfig.DEVICE,
    )

    val_loss, val_m = finetune_evaluate(
        model,
        val_loader,
        criterion,
        FineTuneConfig.DEVICE,
        desc="Fine-tune validation",
    )

    scheduler.step()

    history_ft["train_loss"].append(train_loss)
    history_ft["val_loss"].append(val_loss)
    history_ft["train_f1"].append(train_m["f1"])
    history_ft["val_f1"].append(val_m["f1"])
    history_ft["val_precision"].append(val_m["precision"])
    history_ft["val_recall"].append(val_m["recall"])

    epoch_time = time.perf_counter() - epoch_start
    epoch_times.append(epoch_time)

    avg_epoch_time = sum(epoch_times) / len(epoch_times)
    eta_seconds = avg_epoch_time * (FineTuneConfig.NUM_EPOCHS - epoch)
    elapsed_seconds = time.perf_counter() - total_start

    print(
        f"FineTune Epoch {epoch:02d}/{FineTuneConfig.NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val F1: {val_m['f1']:.4f} | "
        f"Val Exact: {val_m['exact_match']:.4f} | "
        f"Epoch Time: {epoch_time / 60:.1f} min | "
        f"Elapsed: {elapsed_seconds / 60:.1f} min | "
        f"ETA: {eta_seconds / 60:.1f} min"
    )

    improved = (
        val_m["f1"] > best_val_f1
        or (
            abs(val_m["f1"] - best_val_f1) < 1e-6
            and val_loss < best_val_loss
        )
    )

    if improved:
        best_val_f1 = val_m["f1"]
        best_val_loss = val_loss
        torch.save(model.state_dict(), FineTuneConfig.BEST_MODEL_PATH)
        patience_ctr = 0
        print("  Saved new best fine-tuned model.")
    else:
        patience_ctr += 1
        print(f"  No improvement. Patience: {patience_ctr}/{FineTuneConfig.PATIENCE}")

        if patience_ctr >= FineTuneConfig.PATIENCE:
            print(f"Early stopping at fine-tune epoch {epoch}")
            break


# ─────────────────────────────────────────────
# 8. TEST EVALUATION
# ─────────────────────────────────────────────
print("\n── Fine-tuned Test Set Evaluation ──────────────────────────────")

model.load_state_dict(
    torch.load(
        FineTuneConfig.BEST_MODEL_PATH,
        map_location=FineTuneConfig.DEVICE,
    )
)

test_loss, test_m = finetune_evaluate(
    model,
    test_loader,
    criterion,
    FineTuneConfig.DEVICE,
    desc="Fine-tuned test",
)

print(f"Fine-tuned Test Loss:        {test_loss:.4f}")
print(f"Fine-tuned Test F1:          {test_m['f1']:.4f}")
print(f"Fine-tuned Test Precision:   {test_m['precision']:.4f}")
print(f"Fine-tuned Test Recall:      {test_m['recall']:.4f}")
print(f"Fine-tuned Test Exact Match: {test_m['exact_match']:.4f}")

metrics_payload = {
    "model_name": "xenia_resnet50_finetuned_layer3_layer4_head",
    "seed": FineTuneConfig.SEED,
    "img_size": FineTuneConfig.IMG_SIZE,
    "train_size": len(train_df),
    "val_size": len(val_df),
    "test_size": len(test_df),
    "best_val_f1": float(best_val_f1),
    "test_f1": float(test_m["f1"]),
    "test_precision": float(test_m["precision"]),
    "test_recall": float(test_m["recall"]),
    "test_exact_match": float(test_m["exact_match"]),
    "num_epochs_run": int(epochs_run),
    "test_loss": float(test_loss),
    "best_val_loss": float(best_val_loss),
    "notes": (
        "Second-stage fine-tuning from xenia_best_resnet50_fast.pt. "
        "Unfroze ResNet layer3, layer4, and classifier head with smaller learning rates. "
        "Selected best checkpoint by validation F1, with validation loss as tie-breaker."
    ),
}

save_metrics_json(FineTuneConfig.METRICS_PATH, metrics_payload)

print("\nSaved fine-tuned model to:", FineTuneConfig.BEST_MODEL_PATH)
print("Saved fine-tuned metrics to:", FineTuneConfig.METRICS_PATH)
print("Done.")

Using device: cuda
Tesla T4
Previous checkpoint exists: True
Previous checkpoint: /content/drive/MyDrive/ieor142b/results/xenia_best_resnet50_fast.pt
Train batches: 272
Val batches: 34
Test batches: 34
Loaded previous best fast model.
Trainable params: 23,124,504
Total params: 24,569,432

── Fine-tuning ResNet-50: layer3 + layer4 + head ─────────────────


/tmp/ipykernel_21941/2746428859.py:158: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


Fine-tune training:   0%|          | 0/272 [00:00<?, ?it/s]

/tmp/ipykernel_21941/2746428859.py:177: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Fine-tune validation:   0%|          | 0/34 [00:00<?, ?it/s]

...

Fine-tune training:   0%|          | 0/272 [00:00<?, ?it/s]

Fine-tune validation:   0%|          | 0/34 [00:00<?, ?it/s]

FineTune Epoch 03/6 | Train Loss: 0.0097 | Val Loss: 0.0287 | Val F1: 0.4090 | Val Exact: 0.1113 | Epoch Time: 0.9 min | Elapsed: 2.8 min | ETA: 2.8 min
  No improvement. Patience: 2/2
Early stopping at fine-tune epoch 3

── Fine-tuned Test Set Evaluation ──────────────────────────────


Fine-tuned test:   0%|          | 0/34 [00:00<?, ?it/s]

Fine-tuned Test Loss:        0.0309
Fine-tuned Test F1:          0.4037
Fine-tuned Test Precision:   0.5460
Fine-tuned Test Recall:      0.3557
Fine-tuned Test Exact Match: 0.1085

Saved fine-tuned model to: /content/drive/MyDrive/ieor142b/results/xenia_best_resnet50_finetuned.pt
Saved fine-tuned metrics to: /content/drive/MyDrive/ieor142b/results/xenia_resnet50_finetuned_metrics.json
Done.


In [9]:
# ============================================================
# THRESHOLD TUNING FOR FINE-TUNED RESNET-50
# Goal: improve F1 by choosing better threshold than default 0.5
# ============================================================

import json
import numpy as np
import torch
from pathlib import Path
from tqdm.auto import tqdm


# ─────────────────────────────────────────────
# 1. Resolve paths / device
# ─────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

FINE_TUNED_MODEL_PATH = PROJECT_ROOT / "results" / "xenia_best_resnet50_finetuned.pt"
TUNED_METRICS_PATH = PROJECT_ROOT / "results" / "xenia_resnet50_finetuned_threshold_tuned_metrics.json"

print("Using device:", DEVICE)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
print("Fine-tuned checkpoint:", FINE_TUNED_MODEL_PATH)
print("Checkpoint exists:", FINE_TUNED_MODEL_PATH.exists())


# ─────────────────────────────────────────────
# 2. Load fine-tuned model checkpoint
# ─────────────────────────────────────────────
model = PosterGenreClassifier(num_genres).to(DEVICE)

model.load_state_dict(
    torch.load(
        FINE_TUNED_MODEL_PATH,
        map_location=DEVICE,
    )
)

model.eval()
print("Loaded fine-tuned model.")


# ─────────────────────────────────────────────
# 3. Collect logits + labels
# ─────────────────────────────────────────────
@torch.no_grad()
def collect_logits_targets_and_loss(model, loader, criterion, device, desc):
    model.eval()

    all_logits = []
    all_targets = []
    total_loss = 0.0

    for imgs, labels in tqdm(loader, desc=desc, leave=True):
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=(device == "cuda")):
            logits = model(imgs)
            loss = criterion(logits, labels)

        total_loss += loss.item() * imgs.size(0)

        all_logits.append(logits.cpu())
        all_targets.append(labels.cpu())

    avg_loss = total_loss / len(loader.dataset)

    return torch.cat(all_logits), torch.cat(all_targets), avg_loss


criterion = FocalLoss(
    alpha=BASELINE["focal_alpha"],
    gamma=BASELINE["focal_gamma"],
)

val_logits, val_targets, val_loss = collect_logits_targets_and_loss(
    model,
    val_loader,
    criterion,
    DEVICE,
    desc="Collecting validation logits",
)

test_logits, test_targets, test_loss = collect_logits_targets_and_loss(
    model,
    test_loader,
    criterion,
    DEVICE,
    desc="Collecting test logits",
)


# ─────────────────────────────────────────────
# 4. Metric function at custom threshold
# ─────────────────────────────────────────────
def compute_metrics_at_threshold(logits, targets, threshold):
    probs = torch.sigmoid(logits)
    preds = (probs >= threshold).float()

    tp = (preds * targets).sum(dim=1)
    fp = (preds * (1 - targets)).sum(dim=1)
    fn = ((1 - preds) * targets).sum(dim=1)

    precision = (tp / (tp + fp + 1e-8)).mean().item()
    recall = (tp / (tp + fn + 1e-8)).mean().item()
    f1 = (2 * tp / (2 * tp + fp + fn + 1e-8)).mean().item()
    exact = (preds == targets).all(dim=1).float().mean().item()

    avg_predicted_labels = preds.sum(dim=1).float().mean().item()
    avg_true_labels = targets.sum(dim=1).float().mean().item()

    return {
        "threshold": float(threshold),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "exact_match": float(exact),
        "avg_predicted_labels": float(avg_predicted_labels),
        "avg_true_labels": float(avg_true_labels),
    }


# ─────────────────────────────────────────────
# 5. Search best global threshold on validation set
# ─────────────────────────────────────────────
thresholds = np.arange(0.10, 0.71, 0.01)

val_results = []

for threshold in thresholds:
    metrics = compute_metrics_at_threshold(
        val_logits,
        val_targets,
        threshold,
    )
    val_results.append(metrics)

best_val_result = max(val_results, key=lambda x: x["f1"])
best_threshold = best_val_result["threshold"]

print("\nBest validation threshold:")
print(best_val_result)


# Show top 10 thresholds by validation F1
top_10 = sorted(val_results, key=lambda x: x["f1"], reverse=True)[:10]

print("\nTop 10 validation thresholds:")
for r in top_10:
    print(
        f"threshold={r['threshold']:.2f} | "
        f"F1={r['f1']:.4f} | "
        f"Precision={r['precision']:.4f} | "
        f"Recall={r['recall']:.4f} | "
        f"Exact={r['exact_match']:.4f} | "
        f"Avg predicted labels={r['avg_predicted_labels']:.2f}"
    )


# ─────────────────────────────────────────────
# 6. Evaluate test set using best validation threshold
# ─────────────────────────────────────────────
test_default_05 = compute_metrics_at_threshold(
    test_logits,
    test_targets,
    0.50,
)

test_tuned = compute_metrics_at_threshold(
    test_logits,
    test_targets,
    best_threshold,
)

print("\nTest metrics at default threshold 0.50:")
print(test_default_05)

print(f"\nTest metrics at tuned threshold {best_threshold:.2f}:")
print(test_tuned)

print("\nComparison:")
print(f"Test Loss:          {test_loss:.4f}")
print(f"F1:                 {test_default_05['f1']:.4f} → {test_tuned['f1']:.4f}")
print(f"Precision:          {test_default_05['precision']:.4f} → {test_tuned['precision']:.4f}")
print(f"Recall:             {test_default_05['recall']:.4f} → {test_tuned['recall']:.4f}")
print(f"Exact Match:        {test_default_05['exact_match']:.4f} → {test_tuned['exact_match']:.4f}")
print(f"Avg predicted lbls: {test_default_05['avg_predicted_labels']:.2f} → {test_tuned['avg_predicted_labels']:.2f}")


# ─────────────────────────────────────────────
# 7. Save tuned-threshold metrics JSON
# ─────────────────────────────────────────────
metrics_payload = {
    "model_name": "xenia_resnet50_finetuned_threshold_tuned",
    "seed": BASELINE["seed"],
    "img_size": BASELINE["img_size"],
    "train_size": len(train_df),
    "val_size": len(val_df),
    "test_size": len(test_df),
    "best_val_f1": float(best_val_result["f1"]),
    "test_f1": float(test_tuned["f1"]),
    "test_precision": float(test_tuned["precision"]),
    "test_recall": float(test_tuned["recall"]),
    "test_exact_match": float(test_tuned["exact_match"]),
    "num_epochs_run": 0,
    "notes": (
        f"Threshold tuning after fine-tuned ResNet-50. "
        f"Best validation threshold={best_threshold:.2f}. "
        f"Test loss={test_loss:.4f}. "
        f"Default threshold 0.50 test F1={test_default_05['f1']:.4f}; "
        f"tuned threshold test F1={test_tuned['f1']:.4f}."
    ),
    "best_threshold": float(best_threshold),
    "val_loss": float(val_loss),
    "test_loss": float(test_loss),
    "test_default_threshold_0_50": test_default_05,
    "test_tuned_threshold": test_tuned,
    "top_10_validation_thresholds": top_10,
}

save_metrics_json(TUNED_METRICS_PATH, metrics_payload)

print("\nSaved threshold-tuned metrics to:", TUNED_METRICS_PATH)
print("Done.")

Using device: cuda
Tesla T4
Fine-tuned checkpoint: /content/drive/MyDrive/ieor142b/results/xenia_best_resnet50_finetuned.pt
Checkpoint exists: True
Loaded fine-tuned model.


/tmp/ipykernel_21941/673536322.py:58: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device == "cuda")):



Best validation threshold:
{'threshold': 0.3199999999999999, 'precision': 0.5452162027359009, 'recall': 0.5877031683921814, 'f1': 0.5299491882324219, 'exact_match': 0.10395584255456924, 'avg_predicted_labels': 2.597975969314575, 'avg_true_labels': 2.3376264572143555}

Top 10 validation thresholds:
threshold=0.32 | F1=0.5299 | Precision=0.5452 | Recall=0.5877 | Exact=0.1040 | Avg predicted labels=2.60
threshold=0.31 | F1=0.5274 | Precision=0.5320 | Recall=0.5971 | Exact=0.0948 | Avg predicted labels=2.72
threshold=0.33 | F1=0.5272 | Precision=0.5514 | Recall=0.5741 | Exact=0.1058 | Avg predicted labels=2.49
threshold=0.34 | F1=0.5252 | Precision=0.5663 | Recall=0.5538 | Exact=0.1049 | Avg predicted labels=2.32
threshold=0.35 | F1=0.5247 | Precision=0.5893 | Recall=0.5348 | Exact=0.1251 | Avg predicted labels=2.15
threshold=0.30 | F1=0.5244 | Precision=0.5170 | Recall=0.6069 | Exact=0.0902 | Avg predicted labels=2.86
threshold=0.36 | F1=0.5222 | Precision=0.5948 | Recall=0.5244 | Exact=

This is a **big win**. Your best model is now:

```text
Fine-tuned ResNet-50 + threshold = 0.32
Test F1: 0.5027
Test Recall: 0.5697
Test Precision: 0.5065
```

Compared to the original:

```text
Original F1: 0.3885
Final F1:    0.5027
```

That is a very meaningful improvement.

I would **not fine-tune more right now**. The main issue was not that the model was bad; it was that the default threshold `0.5` was too strict. At `0.5`, your model predicted only:

```text
1.21 labels per movie
```

but the true average was:

```text
2.36 labels per movie
```

So it was underpredicting genres. Threshold `0.32` fixes that by predicting:

```text
2.65 labels per movie
```

which is much closer.

The only tradeoff is exact match:

```text
Exact Match: 0.1085 → 0.0901
```

But for multi-label genre classification, **F1 is usually more meaningful than exact match**, because exact match requires every single genre to be perfectly correct.

### What I would do now

Use the threshold-tuned model as your final result:

```text
Model: xenia_resnet50_finetuned_threshold_tuned
Threshold: 0.32
Test F1: 0.5027
Test Precision: 0.5065
Test Recall: 0.5697
Test Exact Match: 0.0901
Test Loss: 0.0309
```

For your writeup, say something like:

> After fine-tuning ResNet-50, threshold tuning on the validation set improved test F1 from 0.4037 at the default 0.5 threshold to 0.5027 at a threshold of 0.32. This suggests that the model was initially underpredicting genres, since the average number of predicted labels increased from 1.21 to 2.65, closer to the true average of 2.36 labels per movie.

If you have extra time, the next experiment I’d try is **per-genre threshold tuning**, where each genre gets its own threshold instead of using one global `0.32`. But if you need a clean final result, stop here and report this model.


In [10]:
# ============================================================
# PER-GENRE THRESHOLD TUNING FOR FINE-TUNED RESNET-50
# ============================================================

import json
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from tqdm.auto import tqdm


# ─────────────────────────────────────────────
# 1. Setup
# ─────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

FINE_TUNED_MODEL_PATH = PROJECT_ROOT / "results" / "xenia_best_resnet50_finetuned.pt"
PER_GENRE_METRICS_PATH = PROJECT_ROOT / "results" / "xenia_resnet50_per_genre_threshold_metrics.json"

print("Using device:", DEVICE)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
print("Fine-tuned checkpoint:", FINE_TUNED_MODEL_PATH)
print("Checkpoint exists:", FINE_TUNED_MODEL_PATH.exists())

class_names = list(mlb.classes_)
num_genres = len(class_names)

print("Num genres:", num_genres)
print("Genres:", class_names)


# ─────────────────────────────────────────────
# 2. Load fine-tuned model
# ─────────────────────────────────────────────
model = PosterGenreClassifier(num_genres).to(DEVICE)

model.load_state_dict(
    torch.load(
        FINE_TUNED_MODEL_PATH,
        map_location=DEVICE,
    )
)

model.eval()
print("Loaded fine-tuned model.")


# ─────────────────────────────────────────────
# 3. Collect logits and targets
# ─────────────────────────────────────────────
criterion = FocalLoss(
    alpha=BASELINE["focal_alpha"],
    gamma=BASELINE["focal_gamma"],
)

@torch.no_grad()
def collect_logits_targets_and_loss(model, loader, criterion, device, desc):
    model.eval()

    all_logits = []
    all_targets = []
    total_loss = 0.0

    for imgs, labels in tqdm(loader, desc=desc, leave=True):
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=(device == "cuda")):
            logits = model(imgs)
            loss = criterion(logits, labels)

        total_loss += loss.item() * imgs.size(0)

        all_logits.append(logits.cpu())
        all_targets.append(labels.cpu())

    avg_loss = total_loss / len(loader.dataset)

    return torch.cat(all_logits), torch.cat(all_targets), avg_loss


val_logits, val_targets, val_loss = collect_logits_targets_and_loss(
    model,
    val_loader,
    criterion,
    DEVICE,
    desc="Collecting validation logits",
)

test_logits, test_targets, test_loss = collect_logits_targets_and_loss(
    model,
    test_loader,
    criterion,
    DEVICE,
    desc="Collecting test logits",
)

val_probs = torch.sigmoid(val_logits)
test_probs = torch.sigmoid(test_logits)

print("Val logits:", val_logits.shape)
print("Test logits:", test_logits.shape)


# ─────────────────────────────────────────────
# 4. Metrics
# ─────────────────────────────────────────────
def sample_metrics_from_preds(preds, targets):
    """
    Same style as your existing sample-averaged metrics.
    """
    preds = preds.float()
    targets = targets.float()

    tp = (preds * targets).sum(dim=1)
    fp = (preds * (1 - targets)).sum(dim=1)
    fn = ((1 - preds) * targets).sum(dim=1)

    precision = (tp / (tp + fp + 1e-8)).mean().item()
    recall = (tp / (tp + fn + 1e-8)).mean().item()
    f1 = (2 * tp / (2 * tp + fp + fn + 1e-8)).mean().item()
    exact = (preds == targets).all(dim=1).float().mean().item()

    avg_predicted_labels = preds.sum(dim=1).float().mean().item()
    avg_true_labels = targets.sum(dim=1).float().mean().item()

    return {
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "exact_match": float(exact),
        "avg_predicted_labels": float(avg_predicted_labels),
        "avg_true_labels": float(avg_true_labels),
    }


def genre_macro_metrics_from_preds(preds, targets):
    """
    Per-genre macro metrics. This treats each genre equally.
    """
    preds = preds.float()
    targets = targets.float()

    tp = (preds * targets).sum(dim=0)
    fp = (preds * (1 - targets)).sum(dim=0)
    fn = ((1 - preds) * targets).sum(dim=0)

    precision_per_genre = tp / (tp + fp + 1e-8)
    recall_per_genre = tp / (tp + fn + 1e-8)
    f1_per_genre = 2 * tp / (2 * tp + fp + fn + 1e-8)

    return {
        "macro_precision": float(precision_per_genre.mean().item()),
        "macro_recall": float(recall_per_genre.mean().item()),
        "macro_f1": float(f1_per_genre.mean().item()),
        "per_genre_precision": precision_per_genre.numpy(),
        "per_genre_recall": recall_per_genre.numpy(),
        "per_genre_f1": f1_per_genre.numpy(),
    }


def preds_with_global_threshold(probs, threshold):
    return (probs >= threshold).float()


def preds_with_per_genre_thresholds(probs, thresholds):
    thresholds_tensor = torch.tensor(thresholds, dtype=probs.dtype).view(1, -1)
    return (probs >= thresholds_tensor).float()


# ─────────────────────────────────────────────
# 5. Tune one threshold per genre on validation
# ─────────────────────────────────────────────
threshold_grid = np.arange(0.05, 0.76, 0.01)

best_thresholds = []
per_genre_rows = []

for j, genre in enumerate(class_names):
    y_true = val_targets[:, j].float()
    y_prob = val_probs[:, j].float()

    best_t = 0.50
    best_f1 = -1.0
    best_precision = 0.0
    best_recall = 0.0

    for t in threshold_grid:
        y_pred = (y_prob >= t).float()

        tp = (y_pred * y_true).sum().item()
        fp = (y_pred * (1 - y_true)).sum().item()
        fn = ((1 - y_pred) * y_true).sum().item()

        precision = tp / (tp + fp + 1e-8)
        recall = tp / (tp + fn + 1e-8)
        f1 = (2 * tp) / (2 * tp + fp + fn + 1e-8)

        if f1 > best_f1:
            best_f1 = f1
            best_t = float(t)
            best_precision = precision
            best_recall = recall

    best_thresholds.append(best_t)

    per_genre_rows.append({
        "genre": genre,
        "threshold": best_t,
        "val_f1": float(best_f1),
        "val_precision": float(best_precision),
        "val_recall": float(best_recall),
        "val_positive_count": int(y_true.sum().item()),
        "val_positive_rate": float(y_true.mean().item()),
    })

threshold_df = pd.DataFrame(per_genre_rows).sort_values(
    by="val_f1",
    ascending=False,
).reset_index(drop=True)

print("\nPer-genre thresholds:")
display(threshold_df)

print("\nThreshold summary:")
print("Mean threshold:", round(float(np.mean(best_thresholds)), 3))
print("Min threshold:", round(float(np.min(best_thresholds)), 3))
print("Max threshold:", round(float(np.max(best_thresholds)), 3))


# ─────────────────────────────────────────────
# 6. Compare validation performance
# ─────────────────────────────────────────────
val_preds_050 = preds_with_global_threshold(val_probs, 0.50)
val_preds_032 = preds_with_global_threshold(val_probs, 0.32)
val_preds_per_genre = preds_with_per_genre_thresholds(val_probs, best_thresholds)

val_sample_050 = sample_metrics_from_preds(val_preds_050, val_targets)
val_sample_032 = sample_metrics_from_preds(val_preds_032, val_targets)
val_sample_per_genre = sample_metrics_from_preds(val_preds_per_genre, val_targets)

val_macro_050 = genre_macro_metrics_from_preds(val_preds_050, val_targets)
val_macro_032 = genre_macro_metrics_from_preds(val_preds_032, val_targets)
val_macro_per_genre = genre_macro_metrics_from_preds(val_preds_per_genre, val_targets)

print("\nVALIDATION SAMPLE-AVERAGED METRICS")
print("Global 0.50:", val_sample_050)
print("Global 0.32:", val_sample_032)
print("Per-genre :", val_sample_per_genre)

print("\nVALIDATION MACRO GENRE METRICS")
print("Global 0.50 macro F1:", round(val_macro_050["macro_f1"], 4))
print("Global 0.32 macro F1:", round(val_macro_032["macro_f1"], 4))
print("Per-genre  macro F1:", round(val_macro_per_genre["macro_f1"], 4))


# ─────────────────────────────────────────────
# 7. Compare test performance
# ─────────────────────────────────────────────
test_preds_050 = preds_with_global_threshold(test_probs, 0.50)
test_preds_032 = preds_with_global_threshold(test_probs, 0.32)
test_preds_per_genre = preds_with_per_genre_thresholds(test_probs, best_thresholds)

test_sample_050 = sample_metrics_from_preds(test_preds_050, test_targets)
test_sample_032 = sample_metrics_from_preds(test_preds_032, test_targets)
test_sample_per_genre = sample_metrics_from_preds(test_preds_per_genre, test_targets)

test_macro_050 = genre_macro_metrics_from_preds(test_preds_050, test_targets)
test_macro_032 = genre_macro_metrics_from_preds(test_preds_032, test_targets)
test_macro_per_genre = genre_macro_metrics_from_preds(test_preds_per_genre, test_targets)

print("\nTEST SAMPLE-AVERAGED METRICS")
print("Global 0.50:", test_sample_050)
print("Global 0.32:", test_sample_032)
print("Per-genre :", test_sample_per_genre)

print("\nTEST MACRO GENRE METRICS")
print("Global 0.50 macro F1:", round(test_macro_050["macro_f1"], 4))
print("Global 0.32 macro F1:", round(test_macro_032["macro_f1"], 4))
print("Per-genre  macro F1:", round(test_macro_per_genre["macro_f1"], 4))

print("\nMAIN TEST COMPARISON")
print(f"Test Loss:              {test_loss:.4f}")
print(f"Sample F1 0.50:         {test_sample_050['f1']:.4f}")
print(f"Sample F1 global 0.32:  {test_sample_032['f1']:.4f}")
print(f"Sample F1 per-genre:    {test_sample_per_genre['f1']:.4f}")
print(f"Macro F1 0.50:          {test_macro_050['macro_f1']:.4f}")
print(f"Macro F1 global 0.32:   {test_macro_032['macro_f1']:.4f}")
print(f"Macro F1 per-genre:     {test_macro_per_genre['macro_f1']:.4f}")


# ─────────────────────────────────────────────
# 8. Per-genre test breakdown
# ─────────────────────────────────────────────
per_genre_test_rows = []

for j, genre in enumerate(class_names):
    per_genre_test_rows.append({
        "genre": genre,
        "threshold": float(best_thresholds[j]),
        "test_precision": float(test_macro_per_genre["per_genre_precision"][j]),
        "test_recall": float(test_macro_per_genre["per_genre_recall"][j]),
        "test_f1": float(test_macro_per_genre["per_genre_f1"][j]),
        "test_positive_count": int(test_targets[:, j].sum().item()),
    })

per_genre_test_df = pd.DataFrame(per_genre_test_rows).sort_values(
    by="test_f1",
    ascending=False,
).reset_index(drop=True)

print("\nPer-genre TEST results using per-genre thresholds:")
display(per_genre_test_df)


# ─────────────────────────────────────────────
# 9. Save metrics JSON
# ─────────────────────────────────────────────
per_genre_thresholds_dict = {
    genre: float(threshold)
    for genre, threshold in zip(class_names, best_thresholds)
}

metrics_payload = {
    "model_name": "xenia_resnet50_finetuned_per_genre_threshold_tuned",
    "seed": BASELINE["seed"],
    "img_size": BASELINE["img_size"],
    "train_size": len(train_df),
    "val_size": len(val_df),
    "test_size": len(test_df),

    # Use sample-averaged F1 as the main leaderboard-style metric
    "best_val_f1": float(val_sample_per_genre["f1"]),
    "test_f1": float(test_sample_per_genre["f1"]),
    "test_precision": float(test_sample_per_genre["precision"]),
    "test_recall": float(test_sample_per_genre["recall"]),
    "test_exact_match": float(test_sample_per_genre["exact_match"]),
    "num_epochs_run": 0,

    "notes": (
        "Per-genre threshold tuning after fine-tuned ResNet-50. "
        "Each genre threshold was selected on the validation set to maximize binary F1 "
        "for that genre. Compared against global thresholds 0.50 and 0.32."
    ),

    "test_loss": float(test_loss),
    "val_loss": float(val_loss),

    "global_threshold_0_50_test": test_sample_050,
    "global_threshold_0_32_test": test_sample_032,
    "per_genre_threshold_test": test_sample_per_genre,

    "global_threshold_0_50_test_macro_f1": float(test_macro_050["macro_f1"]),
    "global_threshold_0_32_test_macro_f1": float(test_macro_032["macro_f1"]),
    "per_genre_threshold_test_macro_f1": float(test_macro_per_genre["macro_f1"]),

    "per_genre_thresholds": per_genre_thresholds_dict,
    "per_genre_validation_table": threshold_df.to_dict(orient="records"),
    "per_genre_test_table": per_genre_test_df.to_dict(orient="records"),
}

save_metrics_json(PER_GENRE_METRICS_PATH, metrics_payload)

print("\nSaved per-genre threshold metrics to:", PER_GENRE_METRICS_PATH)
print("Done.")

Using device: cuda
Tesla T4
Fine-tuned checkpoint: /content/drive/MyDrive/ieor142b/results/xenia_best_resnet50_finetuned.pt
Checkpoint exists: True
Num genres: 24
Genres: ['Action', 'Adventure', 'Animation', 'Biography', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Family', 'Fantasy', 'Film-Noir', 'History', 'Horror', 'Music', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Short', 'Sport', 'Talk-Show', 'Thriller', 'War', 'Western']
Loaded fine-tuned model.


/tmp/ipykernel_21941/1450482914.py:69: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device == "cuda")):


Val logits: torch.Size([1087, 24])
Test logits: torch.Size([1088, 24])

Per-genre thresholds:


,genre,threshold,val_f1,val_precision,val_recall,val_positive_count,val_positive_rate
0,Drama,0.32,0.760855,0.663462,0.891761,619,0.569457
1,Comedy,0.26,0.675248,0.565506,0.837838,407,0.374425
2,Animation,0.30,0.644068,0.703704,0.593750,32,0.029439
3,Action,0.34,0.567657,0.554839,0.581081,148,0.136155
4,Horror,0.46,0.540541,0.666667,0.454545,88,0.080957
5,Romance,0.23,0.481830,0.375262,0.672932,266,0.244710
6,Adventure,0.41,0.464646,0.657143,0.359375,128,0.117755
7,Crime,0.35,0.408163,0.400000,0.416667,168,0.154554
8,Thriller,0.28,0.364865,0.300000,0.465517,116,0.106716
9,Western,0.25,0.357143,0.263158,0.555556,18,0.016559



Threshold summary:
Mean threshold: 0.268
Min threshold: 0.05
Max threshold: 0.46

VALIDATION SAMPLE-AVERAGED METRICS
Global 0.50: {'precision': 0.578043520450592, 'recall': 0.3736583888530731, 'f1': 0.428488165140152, 'exact_match': 0.1168353259563446, 'avg_predicted_labels': 1.2069916725158691, 'avg_true_labels': 2.3376264572143555}
Global 0.32: {'precision': 0.5452162027359009, 'recall': 0.5877031683921814, 'f1': 0.5299491882324219, 'exact_match': 0.10395584255456924, 'avg_predicted_labels': 2.597975969314575, 'avg_true_labels': 2.3376264572143555}
Per-genre : {'precision': 0.4694601595401764, 'recall': 0.6539405584335327, 'f1': 0.512265145778656, 'exact_match': 0.07543697953224182, 'avg_predicted_labels': 3.5216190814971924, 'avg_true_labels': 2.3376264572143555}

VALIDATION MACRO GENRE METRICS
Global 0.50 macro F1: 0.212
Global 0.32 macro F1: 0.2904
Per-genre  macro F1: 0.3393

TEST SAMPLE-AVERAGED METRICS
Global 0.50: {'precision': 0.5459558963775635, 'recall': 0.3556985259056091

,genre,threshold,test_precision,test_recall,test_f1,test_positive_count
0,Drama,0.32,0.628878,0.856911,0.725396,615
1,Comedy,0.26,0.549231,0.832168,0.661724,429
2,Animation,0.30,0.652174,0.500000,0.566038,30
3,Action,0.34,0.451389,0.460993,0.456140,141
4,Romance,0.23,0.315789,0.669643,0.429185,224
5,Horror,0.46,0.588235,0.280374,0.379747,107
6,Thriller,0.28,0.333333,0.397163,0.362460,141
7,Adventure,0.41,0.507463,0.272000,0.354167,125
8,Crime,0.35,0.315476,0.291209,0.302857,182
9,Western,0.25,0.250000,0.350000,0.291667,20



Saved per-genre threshold metrics to: /content/drive/MyDrive/ieor142b/results/xenia_resnet50_per_genre_threshold_metrics.json
Done.


In [ ]:
# TODO Xenia: copy PosterGenreClassifier + train loop from main; swap hyperparams as needed.
# save_metrics_json(_ROOT / "results" / "xenia_resnet_metrics.json", { ... })

Using device: cuda
Poster directory: /content/drive/.shortcut-targets-by-id/16f89WEl9hQQAB3Mo2zOz_FfjHBtB5Q8I/ieor142b/cleaned/downloaded_posters
Poster dir exists: True
Missing train posters: 659
Missing val posters: 78
Missing test posters: 89
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 183MB/s]



── Training Xenia ResNet ──────────────────────────────


: 